# Chroma — 최신 LangChain 통합 방식

이 노트북은 책의 Chroma 예제를 **2026-09-21 기준 공식 LangChain 통합 패턴**으로 다시 구성한 것입니다.

학습 목표는 그대로 유지합니다.

- 메모리/영구 저장 컬렉션 생성
- 문서 추가·수정·삭제와 메타데이터 필터
- similarity/MMR/score-threshold 검색기
- 선택 사항: OpenCLIP을 이용한 이미지 검색과 표준 멀티모달 메시지

주요 변경점:

- `langchain_chroma.Chroma`와 `langchain_text_splitters`를 사용합니다.
- 2026년 종료된 `langchain-community`에는 의존하지 않습니다.
- 책 전용 `langchain_teddynote` 헬퍼를 제거했습니다.
- `load_and_split()` 대신 **load → split** 단계를 명시적으로 분리합니다.
- 문서 ID를 애플리케이션에서 명시해 재실행 시 중복을 방지합니다.
- `persist_directory`를 지정하면 자동으로 영속화되므로 별도 `persist()` 호출은 하지 않습니다.

참고:

- [LangChain Chroma 통합](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)
- [Chroma 문서](https://docs.trychroma.com/)


## 1. 설치

노트북 커널과 같은 환경에 설치하려면 `!pip`보다 `%pip`를 사용합니다. 설치 후 import 오류가 남으면 커널을 한 번 재시작하세요.


In [ ]:
%pip install -qU           "langchain-core>=1,<2" "langchain-openai>=1,<2"           "langchain-text-splitters>=1,<2" "langchain-chroma>=1,<2"           "chromadb>=1.5,<2" python-dotenv


## 2. 환경 변수와 추적

`.env`에는 최소한 `OPENAI_API_KEY`를 둡니다. LangSmith는 선택 사항이며, 키가 있을 때만 추적을 켭니다.

```dotenv
OPENAI_API_KEY="..."
LANGSMITH_API_KEY="..."  # 선택
```


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("LANGSMITH_API_KEY"):
    os.environ.setdefault("LANGSMITH_TRACING", "true")
    os.environ.setdefault("LANGSMITH_PROJECT", "vectorstores-chroma-modern")

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(".env 또는 환경 변수에 OPENAI_API_KEY를 설정하세요.")


In [ ]:
from importlib.metadata import version

for package in ("langchain-core", "langchain-openai", "langchain-chroma", "chromadb"):
    print(f"{package}: {version(package)}")


## 3. 문서 준비

책 저장소의 `data/nlp-keywords.txt`, `data/finance-keywords.txt`가 있으면 그것을 읽고, 없으면 같은 주제의 내장 샘플로 실행합니다. 문서 로딩과 분할을 분리하면 로더가 바뀌어도 분할 정책을 재사용하기 쉽습니다.


In [ ]:
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

data_dir = Path("data")
book_files = [data_dir / "nlp-keywords.txt", data_dir / "finance-keywords.txt"]

if all(path.exists() for path in book_files):
    raw_documents = [
        Document(
            page_content=path.read_text(encoding="utf-8"),
            metadata={"source": path.name, "topic": path.stem.split("-")[0]},
        )
        for path in book_files
    ]
else:
    raw_documents = [
        Document(
            page_content=(
                "TF-IDF는 문서 안의 단어 빈도와 전체 말뭉치의 역문서 빈도를 결합해 "
                "단어의 중요도를 계산한다. 검색과 키워드 추출의 고전적인 기준선이다."
            ),
            metadata={"source": "nlp-keywords.txt", "topic": "nlp"},
        ),
        Document(
            page_content=(
                "Word2Vec은 주변 단어 예측을 통해 단어를 밀집 벡터로 표현한다. "
                "의미가 비슷한 단어는 벡터 공간에서도 가까워지는 경향이 있다."
            ),
            metadata={"source": "nlp-keywords.txt", "topic": "nlp"},
        ),
        Document(
            page_content=(
                "임베딩은 텍스트를 의미를 보존하는 숫자 벡터로 바꾼다. "
                "벡터 검색은 질문과 가까운 문서 조각을 찾는 데 사용된다."
            ),
            metadata={"source": "nlp-keywords.txt", "topic": "nlp"},
        ),
        Document(
            page_content=(
                "ESG는 환경, 사회, 지배구조 관점에서 기업의 지속 가능성을 평가하는 틀이다. "
                "투자자는 재무 정보와 함께 비재무 위험을 살핀다."
            ),
            metadata={"source": "finance-keywords.txt", "topic": "finance"},
        ),
        Document(
            page_content=(
                "채권 가격과 시장 금리는 일반적으로 반대 방향으로 움직인다. "
                "금리가 오르면 기존 고정금리 채권의 상대적 매력은 낮아진다."
            ),
            metadata={"source": "finance-keywords.txt", "topic": "finance"},
        ),
        Document(
            page_content=(
                "분산 투자는 서로 다른 위험 요인을 가진 자산을 함께 보유해 "
                "포트폴리오 전체 변동성을 줄이는 방법이다."
            ),
            metadata={"source": "finance-keywords.txt", "topic": "finance"},
        ),
    ]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
documents = splitter.split_documents(raw_documents)

for index, document in enumerate(documents):
    document.metadata["chunk"] = index

print(f"문서 조각 수: {len(documents)}")
documents[:2]


## 4. 임베딩과 인메모리 Chroma

모델 이름을 명시하면 모델 기본값 변경에 영향을 덜 받습니다. `collection_name`은 Chroma 안에서 컬렉션을 구분하는 이름이며 Pinecone의 namespace와 동일한 개념은 아닙니다.


In [ ]:
from uuid import uuid4

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

collection_name = f"rag-demo-{uuid4().hex[:8]}"
vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
)

document_ids = [f"source-doc-{i:03d}" for i in range(len(documents))]
added_ids = vector_store.add_documents(documents=documents, ids=document_ids)
added_ids[:3]


In [ ]:
# 공개 API로 저장 내용을 확인합니다. ids는 항상 반환되고, include에는 payload만 지정합니다.
snapshot = vector_store.get(include=["documents", "metadatas"])
{key: value[:2] if isinstance(value, list) else value for key, value in snapshot.items()}


### 편의 생성자

이미 문서 목록이 준비되어 있다면 `from_documents()`도 여전히 공식 지원되는 간단한 시작 방법입니다. 반복 실행되는 실습에서는 명시적 ID를 넘기는 것이 안전합니다.


In [ ]:
convenience_store = Chroma.from_documents(
    documents=documents,
    ids=[f"convenience-{i:03d}" for i in range(len(documents))],
    embedding=embeddings,
    collection_name=f"from-documents-{uuid4().hex[:8]}",
)
convenience_store.get(include=["metadatas"])["ids"][:3]


## 5. 로컬 영속화와 다시 연결

최신 Chroma는 `persist_directory`에 변경 사항을 자동 저장합니다. 같은 경로와 같은 컬렉션 이름으로 새 `Chroma` 객체를 만들면 기존 데이터에 다시 연결됩니다.


In [ ]:
persist_directory = Path("chroma_db")
persistent_collection = "rag-demo-persistent"

persistent_store = Chroma(
    collection_name=persistent_collection,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)
persistent_store.add_documents(documents=documents, ids=document_ids)

reloaded_store = Chroma(
    collection_name=persistent_collection,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)
len(reloaded_store.get()["ids"])


## 6. 문서 추가·수정·삭제

신규 문서는 `add_documents()`, 기존 ID의 명시적 수정은 `update_document()`를 사용합니다. ID를 애플리케이션에서 관리하면 업데이트와 삭제가 예측 가능합니다.


In [ ]:
manual_id = "manual-note-001"
vector_store.add_documents(
    documents=[
        Document(
            page_content="벡터 저장소에 새 문서를 추가했습니다.",
            metadata={"source": "manual", "topic": "demo"},
        )
    ],
    ids=[manual_id],
)
vector_store.get(ids=[manual_id], include=["documents", "metadatas"])


In [ ]:
vector_store.update_document(
    document_id=manual_id,
    document=Document(
        page_content="update_document로 기존 문서를 명시적으로 수정했습니다.",
        metadata={"source": "manual", "topic": "demo", "status": "updated"},
    ),
)
vector_store.get(ids=[manual_id], include=["documents", "metadatas"])


In [ ]:
vector_store.delete(ids=[manual_id])
vector_store.get(ids=[manual_id])


## 7. 직접 검색

`similarity_search_with_score()`의 score는 Chroma가 반환하는 **거리 값**입니다. 기본 거리에서는 작을수록 더 가깝습니다. 애플리케이션 임계값은 실제 데이터로 보정하세요.


In [ ]:
query = "TF-IDF는 무엇인가요?"

results = vector_store.similarity_search(query=query, k=3)
for document in results:
    print(document.metadata, "\n", document.page_content, "\n")


In [ ]:
scored_results = vector_store.similarity_search_with_score(query=query, k=3)
for document, distance in scored_results:
    print(f"distance={distance:.4f}", document.metadata, document.page_content)


In [ ]:
# Chroma의 filter는 문서 metadata에 적용됩니다.
vector_store.similarity_search(
    query="임베딩과 단어 벡터",
    k=2,
    filter={"topic": "nlp"},
)


## 8. Retriever로 변환

LangChain의 현재 실행 인터페이스는 `get_relevant_documents()`가 아니라 `invoke()`/`ainvoke()`입니다.

MMR의 `lambda_mult`는 1에 가까울수록 쿼리 유사도를, 0에 가까울수록 결과 다양성을 더 중시합니다.


In [ ]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)
similarity_retriever.invoke("Word2Vec과 임베딩을 설명해 주세요")


In [ ]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 6, "lambda_mult": 0.35},
)
mmr_retriever.invoke("Word2Vec과 임베딩을 설명해 주세요")


In [ ]:
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.5},
)
threshold_retriever.invoke("ESG 투자란 무엇인가요?")


### 컬렉션 초기화

`reset_collection()`은 컬렉션의 모든 레코드를 지우므로 실습 전용 컬렉션에서만 사용합니다.


In [ ]:
scratch_store = Chroma(
    collection_name=f"scratch-{uuid4().hex[:8]}",
    embedding_function=embeddings,
)
scratch_store.add_texts(["곧 삭제할 실습 데이터"], ids=["scratch-1"])
scratch_store.reset_collection()
scratch_store.get()


## 9. 선택 사항 — 멀티모달 이미지 검색

이 절은 모델 가중치와 공개 데이터셋을 내려받으므로 별도로 실행합니다. 실험적 LangChain 래퍼 대신 Chroma가 공식 제공하는 `OpenCLIPEmbeddingFunction`과 `ImageLoader`를 사용해 **텍스트로 이미지를 검색**합니다.


In [ ]:
%pip install -qU           open-clip-torch torch pillow           datasets matplotlib numpy


In [ ]:
from itertools import islice

from datasets import load_dataset
from matplotlib import pyplot as plt

image_directory = Path("tmp/chroma-images")
image_directory.mkdir(parents=True, exist_ok=True)

dataset = load_dataset(
    "detection-datasets/coco",
    split="train",
    streaming=True,
)

image_paths = []
rows = list(islice(dataset, 8))
figure, axes = plt.subplots(2, 4, figsize=(12, 6))
for index, (row, axis) in enumerate(zip(rows, axes.flat)):
    path = image_directory / f"coco-{index:02d}.jpg"
    row["image"].convert("RGB").save(path)
    image_paths.append(path)
    axis.imshow(row["image"])
    axis.axis("off")
plt.tight_layout()


In [ ]:
import chromadb
from chromadb.utils.data_loaders import ImageLoader
from chromadb.utils.embedding_functions import OpenCLIPEmbeddingFunction

multimodal_ef = OpenCLIPEmbeddingFunction(
    model_name="ViT-B-32",
    checkpoint="laion2b_s34b_b79k",
)

multimodal_collection = chromadb.Client().create_collection(
    name=f"multimodal-{uuid4().hex[:8]}",
    embedding_function=multimodal_ef,
    data_loader=ImageLoader(),
)
image_ids = [f"image-{i:02d}" for i in range(len(image_paths))]
multimodal_collection.add(
    ids=image_ids,
    uris=[str(path.resolve()) for path in image_paths],
)


In [ ]:
import base64
from IPython.display import Image as DisplayImage, display

image_results = multimodal_collection.query(
    query_texts=["a dog outdoors"],
    n_results=3,
    include=["uris", "distances"],
)
for uri, distance in zip(image_results["uris"][0], image_results["distances"][0]):
    print(f"distance={distance:.4f}")
    display(DisplayImage(filename=uri, width=240))


### 표준 멀티모달 메시지로 이미지 설명 생성

과거 책 전용 `MultiModal` 래퍼 대신 `ChatOpenAI`에 `HumanMessage`의 텍스트/이미지 content block을 직접 전달합니다. 모델명은 환경 변수로 교체할 수 있습니다.


In [ ]:
import mimetypes

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


def image_data_url(path: Path) -> str:
    mime_type = mimetypes.guess_type(path.name)[0] or "image/jpeg"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime_type};base64,{encoded}"


def message_text(message) -> str:
    if isinstance(message.content, str):
        return message.content
    return " ".join(
        block.get("text", "")
        for block in message.content
        if isinstance(block, dict) and block.get("type") in {"text", "output_text"}
    ).strip()


vision_model = ChatOpenAI(
    model=os.getenv("OPENAI_VISION_MODEL", "gpt-4.1-mini"),
    temperature=0,
)


def describe_image(path: Path) -> str:
    response = vision_model.invoke(
        [
            SystemMessage(content="Describe the image accurately and concisely."),
            HumanMessage(
                content=[
                    {"type": "text", "text": "Write one English sentence under 80 characters."},
                    {
                        "type": "image_url",
                        "image_url": {"url": image_data_url(path)},
                    },
                ]
            ),
        ]
    )
    return message_text(response)


captions = {path: describe_image(path) for path in image_paths[:3]}
captions


In [ ]:
import numpy as np

compared_paths = list(captions)
compared_ids = [image_ids[image_paths.index(path)] for path in compared_paths]
stored = multimodal_collection.get(include=["embeddings"])
vectors_by_id = dict(zip(stored["ids"], stored["embeddings"]))
image_vectors = np.asarray([vectors_by_id[id_] for id_ in compared_ids])
text_vectors = np.asarray(multimodal_ef(list(captions.values())))
cosine_similarity = text_vectors @ image_vectors.T

plt.figure(figsize=(6, 5))
plt.imshow(cosine_similarity, vmin=-1, vmax=1, cmap="coolwarm")
plt.colorbar(label="cosine similarity")
plt.xticks(range(len(compared_paths)), [path.name for path in compared_paths])
plt.yticks(range(len(captions)), list(captions.values()))
plt.tight_layout()


## 정리

- 로컬 실험: 인메모리 `Chroma`
- 재시작 후 재사용: `persist_directory`
- 관리 가능한 갱신: 명시적 ID + `update_document()`/`delete()`
- 체인 연결: `as_retriever()` + `invoke()`
- 멀티모달: Chroma 네이티브 `OpenCLIPEmbeddingFunction` + `ImageLoader`
